# Step 4: Artifact Generation（Hugging Face / Qwen3.5）

这一份 notebook 用冻结后的 Round 1 `source set` 生成两类 memory artifact：

- `episodic_trace.md`
- `cross_episode_consolidation.md`

这一版不再依赖 `OPENAI_API_KEY`，而是默认走本地 `transformers` 推理。

默认模型建议：
- 主推荐：`Qwen/Qwen3.5-9B`
- 显存更稳的回退：`Qwen/Qwen3.5-4B`

路径策略：
- 优先读取环境变量 `SELECT_TRANSFER_ROOT`
- 否则尝试从当前 notebook 所在目录向上定位项目根
- 因此你可以把整个 `2026_SelectTransfer/` 目录上传到云机器后直接运行

当前任务是纯文本 artifact generation，不需要视觉输入；但 `Qwen3.5` 依然可以用 text-only 方式运行。为了减少无关变量，默认关闭 thinking 输出。


In [1]:
import csv
import gc
import json
import os
from pathlib import Path

try:
    import torch
except ImportError:
    torch = None

try:
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
except ImportError:
    AutoModelForCausalLM = None
    AutoTokenizer = None
    BitsAndBytesConfig = None


In [2]:
from pathlib import Path

for p in [
    Path("/workspace/2026_SelectTransfer"),
    Path("/root/2026_SelectTransfer"),
    Path.cwd(),
    Path.cwd().parent,
]:
    print("==", p)
    print("exists:", p.exists())
    if p.exists():
        for child in sorted(p.iterdir()):
            print("  ", child.name)
    print()


== /workspace/2026_SelectTransfer
exists: False

== /root/2026_SelectTransfer
exists: True
   artifacts
   notebooks
   pilot
   results

== /root/2026_SelectTransfer/notebooks
exists: True
   .ipynb_checkpoints
   04_artifact_generation.ipynb

== /root/2026_SelectTransfer
exists: True
   artifacts
   notebooks
   pilot
   results



In [3]:
from pathlib import Path

root = Path("/root/2026_SelectTransfer")
required = [
    "pilot/archive/taxonomy_round1.csv",
    "pilot/archive/source_sets_round1.csv",
    "pilot/archive/pairing_table_round1.csv",
    "results/01_sampling/sampled_20_full.json",
    "results/02_hotpotqa_comparison_expansion/candidate_batch_filtered_full.json",
]

for rel in required:
    p = root / rel
    print(rel, "->", p.exists())


pilot/archive/taxonomy_round1.csv -> True
pilot/archive/source_sets_round1.csv -> True
pilot/archive/pairing_table_round1.csv -> True
results/01_sampling/sampled_20_full.json -> True
results/02_hotpotqa_comparison_expansion/candidate_batch_filtered_full.json -> True


## 1. 配置路径与模型参数


In [4]:
PROJECT_ROOT_OVERRIDE = '/root/2026_SelectTransfer'


def detect_project_root():
    candidates = []

    if PROJECT_ROOT_OVERRIDE.strip():
        candidates.append(Path(PROJECT_ROOT_OVERRIDE).expanduser())

    env_root = os.environ.get('SELECT_TRANSFER_ROOT', '').strip()
    if env_root:
        candidates.append(Path(env_root).expanduser())

    cwd = Path.cwd().resolve()
    candidates.extend([
        cwd,
        cwd.parent,
        cwd / '2026_SelectTransfer',
        Path('/content/2026_SelectTransfer'),
        Path('/workspace/2026_SelectTransfer'),
        Path('/root/2026_SelectTransfer'),
        Path('/kaggle/working/2026_SelectTransfer'),
    ])

    seen = set()
    checked = []
    for candidate in candidates:
        if candidate in seen:
            continue
        seen.add(candidate)
        checked.append(str(candidate))
        if (candidate / 'pilot' / 'archive' / 'taxonomy_round1.csv').exists() and (candidate / 'results' / '01_sampling' / 'sampled_20_full.json').exists():
            return candidate
    raise FileNotFoundError(
        'Could not locate project root. Set PROJECT_ROOT_OVERRIDE or SELECT_TRANSFER_ROOT to the uploaded 2026_SelectTransfer directory. Checked: ' + ' | '.join(checked)
    )


PROJECT_ROOT = detect_project_root()
ARCHIVE_DIR = PROJECT_ROOT / 'pilot' / 'archive'
ARTIFACTS_DIR = PROJECT_ROOT / 'artifacts'
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

TAXONOMY_PATH = ARCHIVE_DIR / 'taxonomy_round1.csv'
SOURCE_SETS_PATH = ARCHIVE_DIR / 'source_sets_round1.csv'
PAIRING_PATH = ARCHIVE_DIR / 'pairing_table_round1.csv'

SAMPLED_JSON_PATH = PROJECT_ROOT / 'results' / '01_sampling' / 'sampled_20_full.json'
EXPANDED_JSON_PATH = PROJECT_ROOT / 'results' / '02_hotpotqa_comparison_expansion' / 'candidate_batch_filtered_full.json'

MODEL_ID = 'Qwen/Qwen3.5-9B'
FALLBACK_MODEL_ID = 'Qwen/Qwen3.5-4B'
HF_TOKEN = os.environ.get('HF_TOKEN', '')
RUN_GENERATION = True
USE_4BIT = False
ENABLE_THINKING = False
MAX_NEW_TOKENS = 1400
DO_SAMPLE = False
TARGET_SOURCE_SET_IDS = None  # 例如 ['hp_bridge_set_01']

print('PROJECT_ROOT =', PROJECT_ROOT)
print('ARTIFACTS_DIR =', ARTIFACTS_DIR)
print('MODEL_ID =', MODEL_ID)
print('FALLBACK_MODEL_ID =', FALLBACK_MODEL_ID)
print('RUN_GENERATION =', RUN_GENERATION)
print('USE_4BIT =', USE_4BIT)
print('ENABLE_THINKING =', ENABLE_THINKING)
print('torch import available =', torch is not None)
if torch is not None and torch.cuda.is_available():
    print('CUDA device =', torch.cuda.get_device_name(0))
    print('CUDA bf16 supported =', torch.cuda.is_bf16_supported())
    total_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print('CUDA total memory (GB) =', round(total_gb, 2))
print('transformers import available =', AutoTokenizer is not None)


PROJECT_ROOT = /root/2026_SelectTransfer
ARTIFACTS_DIR = /root/2026_SelectTransfer/artifacts
MODEL_ID = Qwen/Qwen3.5-9B
FALLBACK_MODEL_ID = Qwen/Qwen3.5-4B
RUN_GENERATION = True
USE_4BIT = False
ENABLE_THINKING = False
torch import available = True
CUDA device = NVIDIA GeForce RTX 4090
CUDA bf16 supported = True
CUDA total memory (GB) = 47.37
transformers import available = True


## 2. 读取冻结后的 Round 1 输入


In [5]:
def read_csv(path):
    with path.open(newline='', encoding='utf-8') as f:
        return list(csv.DictReader(f))


def load_json(path):
    return json.loads(path.read_text(encoding='utf-8'))


taxonomy_rows = read_csv(TAXONOMY_PATH)
source_set_rows = read_csv(SOURCE_SETS_PATH)
pairing_rows = read_csv(PAIRING_PATH)

print('taxonomy rows:', len(taxonomy_rows))
print('source set rows:', len(source_set_rows))
print('pairing rows:', len(pairing_rows))


taxonomy rows: 35
source set rows: 2
pairing rows: 10


## 3. 构造 source task bank


In [6]:
sampled_rows = load_json(SAMPLED_JSON_PATH)
expanded_rows = load_json(EXPANDED_JSON_PATH)
all_payload_rows = sampled_rows + expanded_rows

task_payload_map = {row['task_id']: row for row in all_payload_rows}
taxonomy_map = {row['task_id']: row for row in taxonomy_rows}

missing_payload_task_ids = []
for source_set in source_set_rows:
    for task_id in source_set['member_task_ids'].split('|'):
        task_id = task_id.strip()
        if task_id and task_id not in task_payload_map:
            missing_payload_task_ids.append(task_id)

print('payload rows available:', len(task_payload_map))
print('missing payload task ids:', missing_payload_task_ids)


payload rows available: 35
missing payload task ids: []


## 4. Helper functions：从 frozen source material 构造 task card


In [7]:
def parse_members(cell):
    return [x.strip() for x in str(cell).split('|') if x.strip()]


def extract_support_sentences(raw):
    support = raw.get('supporting_facts', {})
    context = raw.get('context', {})
    titles = context.get('title', []) or []
    sentences = context.get('sentences', []) or []

    extracted = []
    for title, sent_id in zip(support.get('title', []), support.get('sent_id', [])):
        sentence_text = ''
        if title in titles:
            idx = titles.index(title)
            title_sents = sentences[idx]
            if 0 <= sent_id < len(title_sents):
                sentence_text = str(title_sents[sent_id]).strip()
        extracted.append({
            'title': title,
            'sent_id': sent_id,
            'sentence': sentence_text,
        })
    return extracted


def task_card(task_id):
    taxonomy = taxonomy_map[task_id]
    payload = task_payload_map[task_id]
    raw = payload['raw']
    support_entries = extract_support_sentences(raw)
    support_titles = [entry['title'] for entry in support_entries]

    return {
        'task_id': task_id,
        'dataset': taxonomy['dataset'],
        'reasoning_label': taxonomy['reasoning_label'],
        'question': taxonomy['question'],
        'answer': taxonomy['answer'],
        'taxonomy_note': taxonomy['note'],
        'raw_type': raw.get('type', ''),
        'level': raw.get('level', ''),
        'support_titles': support_titles,
        'support_entries': support_entries,
    }


def render_task_card(card):
    lines = []
    lines.append(f"- Task ID: {card['task_id']}")
    lines.append(f"  - Question: {card['question']}")
    lines.append(f"  - Answer: {card['answer']}")
    lines.append(f"  - Reasoning Label: {card['reasoning_label']}")
    lines.append(f"  - Raw Type: {card['raw_type']}")
    lines.append(f"  - Difficulty: {card['level']}")
    lines.append(f"  - Taxonomy Note: {card['taxonomy_note']}")
    if card['support_titles']:
        lines.append(f"  - Supporting Titles: {', '.join(card['support_titles'])}")
    else:
        lines.append('  - Supporting Titles: none extracted')

    nonempty_support = [entry for entry in card['support_entries'] if entry['sentence']]
    if nonempty_support:
        lines.append('  - Supporting Sentences:')
        for entry in nonempty_support:
            lines.append(f"    - [{entry['title']} / sent {entry['sent_id']}] {entry['sentence']}")
    return "\n".join(lines)


def build_source_set_payload(source_set_row):
    member_ids = parse_members(source_set_row['member_task_ids'])
    cards = [task_card(task_id) for task_id in member_ids]
    rendered_cards = "\n\n".join(render_task_card(card) for card in cards)
    return {
        'source_set_id': source_set_row['source_set_id'],
        'cluster': source_set_row['cluster'],
        'note': source_set_row['note'],
        'member_ids': member_ids,
        'cards': cards,
        'rendered_cards': rendered_cards,
    }


## 5. 定义两类 artifact prompt


In [8]:
EPISODIC_SYSTEM_PROMPT = '''You are writing a reusable memory artifact for an LLM agent experiment.

Your job is to compress a source set into an episodic trace that still preserves episode-level specificity.

Hard constraints:
- Output markdown only.
- Preserve all five episodes as separate units.
- Do not turn this into a high-level principle list.
- Do not invent evidence that is not present.
- Do not mention this prompt or the experiment instructions.

The artifact should remain close to solved episodes while being shorter and cleaner than raw trajectories.'''

EPISODIC_USER_TEMPLATE = '''Write `episodic_trace.md` for the following frozen source set.

Source Set ID: {source_set_id}
Cluster: {cluster}
Source Set Note: {source_set_note}

Input episodes:
{rendered_cards}

Required output structure:

# Episodic Trace

## Source Set
- source_set_id: ...
- cluster: ...

## Episode Summaries
### 1. {first_task_id}
- question:
- answer:
- key lookup path:
- minimal support:
- reusable cue:

(repeat for all episodes)

## Local Pattern Notes
- 3 to 5 bullets only

Important distinction:
- keep this artifact episode-grounded
- preserve specific lookup paths and local cues
- do not collapse the whole set into generic advice
'''

CONSOLIDATION_SYSTEM_PROMPT = '''You are writing a reusable memory artifact for an LLM agent experiment.

Your job is to consolidate a source set into a cross-episode memory that captures shared structure, applicability, and boundaries.

Hard constraints:
- Output markdown only.
- Do not repeat all episodes one by one in detail.
- Synthesize across the set.
- Do not invent evidence that is not present.
- Do not mention this prompt or the experiment instructions.

The artifact should read like a compact reusable principle note, not an episode list.'''

CONSOLIDATION_USER_TEMPLATE = '''Write `cross_episode_consolidation.md` for the following frozen source set.

Source Set ID: {source_set_id}
Cluster: {cluster}
Source Set Note: {source_set_note}

Input episodes:
{rendered_cards}

Required output structure:

# Cross-Episode Consolidation

## Source Set
- source_set_id: ...
- cluster: ...

## Shared Structure
- 3 to 5 bullets

## Applicability
- when this memory is likely useful
- when it is not the right memory to use

## Operational Heuristic
- a short ordered checklist for applying this memory form

## Boundary / Failure Risk
- 2 to 4 bullets

Important distinction:
- this artifact must be meaningfully more abstract than `episodic_trace`
- it should emphasize shared structure, applicability, and boundary conditions
- avoid empty advice like "read carefully" or "reason step by step" unless grounded in the episodes above
'''


## 6. 准备本地 Hugging Face model 与 artifact 目录

默认推荐：
- `Qwen/Qwen3.5-9B`：如果 32GB 卡能稳定跑通，优先用它生成 artifact
- `Qwen/Qwen3.5-4B`：如果 9B OOM，直接回退到它

当前默认关闭 `thinking`，避免生成冗长 `<think>` 内容，减少无关变量。


In [9]:
def infer_torch_dtype():
    if torch is None:
        raise ImportError('torch is not available. Install torch before local generation.')
    if torch.cuda.is_available():
        if torch.cuda.is_bf16_supported():
            return torch.bfloat16
        return torch.float16
    return torch.float32


def load_local_model(model_id):
    if torch is None:
        raise ImportError('torch is not available. Run this notebook in Colab or another GPU Python environment.')
    if AutoTokenizer is None or AutoModelForCausalLM is None:
        raise ImportError('transformers is not available. Run the install cell first.')

    token = HF_TOKEN or None
    tokenizer = AutoTokenizer.from_pretrained(model_id, token=token)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model_kwargs = {
        'device_map': 'auto',
        'token': token,
    }
    torch_dtype = infer_torch_dtype()

    if USE_4BIT:
        if BitsAndBytesConfig is None:
            raise ImportError('BitsAndBytesConfig is not available. Install bitsandbytes or set USE_4BIT = False.')
        compute_dtype = torch_dtype if torch_dtype != torch.float32 else torch.float16
        model_kwargs['quantization_config'] = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=compute_dtype,
            bnb_4bit_quant_type='nf4',
            bnb_4bit_use_double_quant=True,
        )
    else:
        model_kwargs['torch_dtype'] = torch_dtype

    model = AutoModelForCausalLM.from_pretrained(model_id, **model_kwargs)
    model.eval()
    return tokenizer, model, torch_dtype


source_set_payloads = []
for row in source_set_rows:
    if TARGET_SOURCE_SET_IDS is not None and row['source_set_id'] not in TARGET_SOURCE_SET_IDS:
        continue
    payload = build_source_set_payload(row)
    source_set_payloads.append(payload)
    (ARTIFACTS_DIR / row['source_set_id']).mkdir(parents=True, exist_ok=True)

print('source sets selected for artifact generation:')
for payload in source_set_payloads:
    print('-', payload['source_set_id'])

tokenizer = None
model = None
torch_dtype = None
if RUN_GENERATION:
    tokenizer, model, torch_dtype = load_local_model(MODEL_ID)
    print('Loaded model =', MODEL_ID)
    print('Loaded dtype =', torch_dtype)
else:
    print('RUN_GENERATION = False -> prompt preview only')


source sets selected for artifact generation:
- hp_bridge_set_01
- hp_comparison_set_01


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/12.8M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/427 [00:00<?, ?it/s]

Loaded model = Qwen/Qwen3.5-9B
Loaded dtype = torch.bfloat16


## 7. 先写 prompt preview，再决定是否正式生成


In [10]:
def build_episodic_user_prompt(payload):
    return EPISODIC_USER_TEMPLATE.format(
        source_set_id=payload['source_set_id'],
        cluster=payload['cluster'],
        source_set_note=payload['note'],
        rendered_cards=payload['rendered_cards'],
        first_task_id=payload['member_ids'][0],
    )


def build_consolidation_user_prompt(payload):
    return CONSOLIDATION_USER_TEMPLATE.format(
        source_set_id=payload['source_set_id'],
        cluster=payload['cluster'],
        source_set_note=payload['note'],
        rendered_cards=payload['rendered_cards'],
    )


for payload in source_set_payloads:
    out_dir = ARTIFACTS_DIR / payload['source_set_id']
    episodic_prompt = build_episodic_user_prompt(payload)
    consolidation_prompt = build_consolidation_user_prompt(payload)

    (out_dir / 'prompt_episodic_trace.md').write_text(episodic_prompt, encoding='utf-8')
    (out_dir / 'prompt_cross_episode_consolidation.md').write_text(consolidation_prompt, encoding='utf-8')

print('Prompt previews written under artifacts/<source_set_id>/.')


Prompt previews written under artifacts/<source_set_id>/.


## 8. 用本地 `transformers` 生成 artifact（或保持 prompt-only 模式）


In [11]:
def build_chat_text(system_prompt, user_prompt):
    messages = [
        {'role': 'system', 'content': system_prompt},
        {'role': 'user', 'content': user_prompt},
    ]
    try:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=ENABLE_THINKING,
        )
    except TypeError:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )


def get_model_device(model_obj):
    return next(model_obj.parameters()).device


def generate_markdown(system_prompt, user_prompt):
    if torch is None:
        raise ImportError('torch is not available. Local generation requires a GPU Python environment.')
    prompt_text = build_chat_text(system_prompt, user_prompt)
    model_inputs = tokenizer([prompt_text], return_tensors='pt')
    model_device = get_model_device(model)
    model_inputs = {k: v.to(model_device) for k, v in model_inputs.items()}

    with torch.inference_mode():
        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=DO_SAMPLE,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    output_ids = generated_ids[0][model_inputs['input_ids'].shape[1]:]
    text = tokenizer.decode(output_ids, skip_special_tokens=True).strip()

    if not ENABLE_THINKING:
        text = text.replace('<think>', '').replace('</think>', '').strip()

    return text, {'prompt_chars': len(prompt_text)}


generation_manifest = []

for payload in source_set_payloads:
    out_dir = ARTIFACTS_DIR / payload['source_set_id']
    episodic_user_prompt = build_episodic_user_prompt(payload)
    consolidation_user_prompt = build_consolidation_user_prompt(payload)

    if not RUN_GENERATION:
        generation_manifest.append({
            'source_set_id': payload['source_set_id'],
            'artifact_type': 'episodic_trace',
            'status': 'prompt_only',
            'model': MODEL_ID,
            'backend': 'huggingface_transformers',
            'output_file': str(out_dir / 'episodic_trace.md'),
            'char_count': '',
            'error': '',
        })
        generation_manifest.append({
            'source_set_id': payload['source_set_id'],
            'artifact_type': 'cross_episode_consolidation',
            'status': 'prompt_only',
            'model': MODEL_ID,
            'backend': 'huggingface_transformers',
            'output_file': str(out_dir / 'cross_episode_consolidation.md'),
            'char_count': '',
            'error': '',
        })
        continue

    for artifact_type, system_prompt, user_prompt, filename in [
        ('episodic_trace', EPISODIC_SYSTEM_PROMPT, episodic_user_prompt, 'episodic_trace.md'),
        ('cross_episode_consolidation', CONSOLIDATION_SYSTEM_PROMPT, consolidation_user_prompt, 'cross_episode_consolidation.md'),
    ]:
        try:
            text, meta = generate_markdown(system_prompt, user_prompt)
            (out_dir / filename).write_text(text + "\n", encoding='utf-8')
            generation_manifest.append({
                'source_set_id': payload['source_set_id'],
                'artifact_type': artifact_type,
                'status': 'generated',
                'model': MODEL_ID,
                'backend': 'huggingface_transformers',
                'output_file': str(out_dir / filename),
                'char_count': len(text),
                'error': '',
            })
        except Exception as e:
            generation_manifest.append({
                'source_set_id': payload['source_set_id'],
                'artifact_type': artifact_type,
                'status': 'error',
                'model': MODEL_ID,
                'backend': 'huggingface_transformers',
                'output_file': str(out_dir / filename),
                'char_count': '',
                'error': str(e)[:300],
            })
            print('Generation error:', payload['source_set_id'], artifact_type, e)

manifest_path = ARTIFACTS_DIR / 'round1_artifact_generation_manifest.csv'
fieldnames = ['source_set_id', 'artifact_type', 'status', 'model', 'backend', 'output_file', 'char_count', 'error']
with manifest_path.open('w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    for row in generation_manifest:
        writer.writerow(row)

print(f'Wrote manifest to: {manifest_path}')
for row in generation_manifest:
    print(row)

if model is not None:
    gc.collect()
    if torch is not None and torch.cuda.is_available():
        torch.cuda.empty_cache()


Wrote manifest to: /root/2026_SelectTransfer/artifacts/round1_artifact_generation_manifest.csv
{'source_set_id': 'hp_bridge_set_01', 'artifact_type': 'episodic_trace', 'status': 'generated', 'model': 'Qwen/Qwen3.5-9B', 'backend': 'huggingface_transformers', 'output_file': '/root/2026_SelectTransfer/artifacts/hp_bridge_set_01/episodic_trace.md', 'char_count': 3161, 'error': ''}
{'source_set_id': 'hp_bridge_set_01', 'artifact_type': 'cross_episode_consolidation', 'status': 'generated', 'model': 'Qwen/Qwen3.5-9B', 'backend': 'huggingface_transformers', 'output_file': '/root/2026_SelectTransfer/artifacts/hp_bridge_set_01/cross_episode_consolidation.md', 'char_count': 3100, 'error': ''}
{'source_set_id': 'hp_comparison_set_01', 'artifact_type': 'episodic_trace', 'status': 'generated', 'model': 'Qwen/Qwen3.5-9B', 'backend': 'huggingface_transformers', 'output_file': '/root/2026_SelectTransfer/artifacts/hp_comparison_set_01/episodic_trace.md', 'char_count': 3787, 'error': ''}
{'source_set_id'

## 9. 本地快速检查输出


In [12]:
for payload in source_set_payloads:
    out_dir = ARTIFACTS_DIR / payload['source_set_id']
    print('=' * 80)
    print(payload['source_set_id'])
    for filename in ['episodic_trace.md', 'cross_episode_consolidation.md']:
        path = out_dir / filename
        print('-' * 40)
        print(filename, 'exists =', path.exists())
        if path.exists():
            text = path.read_text(encoding='utf-8')
            print(text[:1200])
            print()


hp_bridge_set_01
----------------------------------------
episodic_trace.md exists = True
# Episodic Trace

## Source Set
- source_set_id: hp_bridge_set_01
- cluster: bridge

## Episode Summaries
### 1. hp_dev_2054
- question: When did the park at which Tivolis Koncertsal is located open?
- answer: 15 August 1843
- key lookup path: Tivolis Koncertsal -> location (Tivoli Gardens) -> park opening date
- minimal support: Tivolis Koncertsal is a concert hall located at Tivoli Gardens in Copenhagen, Denmark.
- reusable cue: Identify the specific venue's parent park or garden entity to retrieve the establishment date.

### 2. hp_dev_3245
- question: The school in which the Wilmslow Show is held is designated as what?
- answer: Centre of Excellence
- key lookup path: Wilmslow Show -> venue (Wilmslow High School) -> school designation
- minimal support: Wilmslow Show is held at Wilmslow High School; Wilmslow High School is a designated Centre of Excellence.
- reusable cue: Trace the event to i

## 10. 进入下一步前的人工 review 提醒

在进入 `pilot run` 之前，至少确认：

- `episodic_trace` 和 `cross_episode_consolidation` 读起来有实质差异
- `cross_episode_consolidation` 不是空泛 advice
- 两类 artifact 都没有泄漏 target 答案
- 如果 `Qwen/Qwen3.5-9B` 在 32GB 卡上 OOM，先切到 `Qwen/Qwen3.5-4B`
- 如果还要保留 `Qwen3.5-9B`，再考虑 `USE_4BIT = True`，不要同时改太多变量
